# Scores, thresholds and rare classes

This notebook covers the second block of Session 8. The churn model of the first block produces probabilities; the retention team of the telecom company has to decide each month which customers receive a retention offer, and the customers it most wants to find (the 26.5 % who leave) are the minority. The page first evaluates the scores over all thresholds (the ROC curve with its AUC and the precision–recall curve), then chooses a **decision threshold** from the costs of the two kinds of error, compares the usual remedies for **class imbalance** (class weights, undersampling, oversampling and SMOTE, applied inside the cross-validation only), and ends by checking whether the predicted probabilities can be trusted (**calibration**).

The code cells build on each other; run them in order. The first cell moves to the repository root, where the case-study paths start. The first code block rebuilds the Telco pipelines of lecture notebook 01. The imbalance sections need the package `imbalanced-learn` (imported as `imblearn`), which is part of the course environment.

```mermaid
flowchart LR
    M["Fitted classifier"] --> P["Probabilities<br/>predict_proba"]
    P --> R["Ranking quality:<br/>ROC AUC, PR curve"]
    P --> CAL{"Calibrated?<br/>(calibration curve)"}
    CAL -->|no| FIX["Recalibrate:<br/>CalibratedClassifierCV"]
    CAL -->|yes| TH["Choose threshold<br/>from error costs<br/>(on validation data)"]
    FIX --> TH
    TH --> D["Decision:<br/>offer / no offer"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## The ROC curve and AUC

**Concept.** Most classifiers output a score or probability; a **threshold** turns it into a class (by default 0.5). Each threshold gives one pair of

- **true positive rate** (TPR, recall) = TP / (TP + FN), and
- **false positive rate** (FPR) = FP / (FP + TN), the share of loyal customers wrongly flagged.

The **ROC curve** (receiver operating characteristic) plots TPR against FPR for all thresholds, from "flag nobody" (0, 0) to "flag everybody" (1, 1). The **area under the curve** (ROC AUC) summarises the ranking quality in one number. It equals the probability that a randomly chosen churner gets a higher score than a randomly chosen non-churner. 0.5 is random guessing; 1.0 is a perfect ranking. AUC does not depend on any threshold.

Worked example: four customers with scores 0.9 (churns), 0.6 (stays), 0.4 (churns), 0.2 (stays). Of the 2 × 2 = 4 (churner, non-churner) pairs, the churner has the higher score in 3: (0.9 > 0.6), (0.9 > 0.2), (0.4 > 0.2); not in (0.4 < 0.6). AUC = 3/4 = 0.75.

![ROC curves (left) and precision–recall curves (right) of logistic regression and 50-NN on the Telco test customers. The dot marks the default threshold 0.5 of logistic regression.](../figures/roc_pr_curves.png)

**Why it matters.** AUC compares models independently of the threshold, which is often decided later by the business (threshold section below). It is also insensitive to the class proportions, which makes it comparable across datasets.

**How it works in Python.** ROC and AUC need **scores**, not predicted classes: use `predict_proba(X)[:, 1]` (or `decision_function`).

In [2]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# the Telco data, split and pipelines of lecture notebook 01
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
telco = pd.read_csv(URL)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")
y = (telco["Churn"] == "Yes").astype(int)
X = telco.drop(columns=["customerID", "Churn"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
numeric = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
categorical = [c for c in X.columns if c not in numeric]
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])
logreg = make_pipeline(pre, LogisticRegression(max_iter=1000)).fit(X_train, y_train)
knn = make_pipeline(pre, KNeighborsClassifier(n_neighbors=50)).fit(X_train, y_train)

from sklearn.metrics import roc_auc_score, roc_curve

proba_lr = logreg.predict_proba(X_test)[:, 1]
proba_knn = knn.predict_proba(X_test)[:, 1]
print(round(roc_auc_score(y_test, proba_lr), 3), round(roc_auc_score(y_test, proba_knn), 3))   # 0.844 0.834

fpr, tpr, thresholds = roc_curve(y_test, proba_lr)
i = np.argmin(np.abs(thresholds - 0.5))
print(round(fpr[i], 3), round(tpr[i], 3))     # about 0.114 0.531: the dot on the curve at threshold 0.5

print(roc_auc_score([1, 0, 1, 0], [0.9, 0.6, 0.4, 0.2]))   # 0.75, the worked example

0.844 0.834
0.114 0.531
0.75


`RocCurveDisplay.from_estimator(model, X_test, y_test)` draws the curve. The scikit-learn gallery notebook [extras/06-roc-curve.ipynb](extras/06-roc-curve.ipynb) extends ROC to several classes (one-vs-rest and one-vs-one, micro- and macro-averaged).

**In practice.**
- ROC analysis comes from signal detection for radar in the Second World War and is standard in diagnostic medicine for comparing tests (Hanley & McNeil, 1982).
- Credit scoring reports the Gini coefficient, which is 2 · AUC − 1, as the standard measure of a scorecard's discriminatory power.

> **Caution.** Passing predicted classes instead of probabilities to `roc_auc_score` gives a single-point "curve" and a misleading AUC. Use `predict_proba(X)[:, 1]`.

> **Warning.** When the positive class is very rare (fraud, 0.1 %), the FPR is computed over a huge number of negatives, so even many false alarms look like a small FPR. The ROC curve can then look excellent while precision is poor. Use the precision–recall curve as well.

## The precision–recall curve

**Concept.** The **precision–recall (PR) curve** plots precision against recall for all thresholds. At high thresholds few customers are flagged, mostly correctly (high precision, low recall); lowering the threshold moves to the right (more recall, lower precision). A random ranking has precision equal to the share of positives (here 0.265) at every recall. **Average precision** (AP) summarises the curve as a weighted mean of the precisions, using the recall increases as weights.

**Why it matters.** The PR curve focuses on the positive class and does not use the true negatives at all. When positives are rare and the business works through a ranked list (call the top 500 customers, review the top 100 transactions), precision at a given recall is the number that matters. The baseline of the PR curve also makes the difficulty of the task visible: an AP of 0.64 with a baseline of 0.265 is a clear gain.

**How it works in Python.**

In [3]:
from sklearn.metrics import average_precision_score, precision_recall_curve

print(round(average_precision_score(y_test, proba_lr), 3), round(average_precision_score(y_test, proba_knn), 3))   # 0.638 0.617
print(round(y_test.mean(), 3))                         # 0.265: AP of a random ranking

precision, recall, thr = precision_recall_curve(y_test, proba_lr)
ok = recall[:-1] >= 0.8                                # thresholds that find at least 80 % of churners
best = np.argmax(np.where(ok, precision[:-1], 0))
print(round(thr[best], 3), round(precision[best], 3), round(recall[best], 3))
# 0.268 0.524 0.807: to find 80 % of churners, almost every second flagged customer is a false alarm

0.638 0.617
0.265
0.268 0.524 0.807


`PrecisionRecallDisplay.from_estimator` draws the curve; the scikit-learn gallery notebook [extras/07-precision-recall-curve.ipynb](extras/07-precision-recall-curve.ipynb) covers the multiclass case.

**In practice.**
- Fraud detection and anti-money-laundering systems are evaluated by precision at the alert volume analysts can handle, because the positive class is a fraction of a percent.
- Object detection benchmarks such as PASCAL VOC and COCO rank models by mean average precision (mAP) computed from precision–recall curves.

> **Note.** Which metric fits? If the business question is "rank customers by risk", report ROC AUC and AP. If it is "who gets an offer this month", report precision and recall at the chosen threshold, ideally with the cost of errors (next section).

## Decision thresholds from the costs of errors

**Concept.** `predict` uses a threshold of 0.5: flag a customer if the predicted churn probability is at least 0.5. That choice assumes that a false positive and a false negative cost the same, which is rarely true. Suppose

- every flagged customer receives a retention offer costing **c = €100**, and the offer keeps a churner;
- every churner who is not flagged is lost, costing **L = €400** in future margin.

If a customer has churn probability *p*, flagging costs €100 for sure; not flagging costs €400 with probability *p*, so €400 · *p* on average. Flagging is cheaper when 400 · *p* > 100, that is when **p > c / L = 0.25**. The cost-optimal threshold is 0.25, not 0.5. This simple rule holds only if the probabilities are **calibrated** (next section); in practice we also check it empirically: compute the total cost for a grid of thresholds on **validation** data and take the minimum.

![Cost per customer against the threshold for three assumed costs of a lost customer (€200, €400, €800) and an offer of €100, from out-of-fold probabilities of the logistic regression. Each optimum lies close to c / L; the dashed line is the default 0.5.](../figures/cost_threshold.png)

**Why it matters.** The threshold turns a model score into a decision that can be justified to the business. Moving it changes precision and recall along the curves of the first two sections; the costs decide where on the curve to operate. A model with a good AUC but a badly chosen threshold can cost more than a simple rule.

**How it works in Python.** Out-of-fold probabilities from `cross_val_predict` on the training data give an honest validation set for choosing the threshold; the test set is used only to check the choice.

In [4]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

OFFER, LOST = 100, 400                         # € per offer, € per lost churner

def cost(y_true, proba, t):
    flagged = proba >= t
    return OFFER * flagged.sum() + LOST * (~flagged & (np.asarray(y_true) == 1)).sum()

# choose the threshold on out-of-fold probabilities of the TRAINING data
oof = cross_val_predict(logreg, X_train, y_train, method="predict_proba",
                        cv=StratifiedKFold(5, shuffle=True, random_state=0))[:, 1]
ts = np.round(np.arange(0.05, 0.96, 0.05), 2)
costs = [cost(y_train, oof, t) for t in ts]
t_best = ts[np.argmin(costs)]
print(t_best, min(costs), cost(y_train, oof, 0.5))     # 0.25 329800 367500  (5,282 training customers)

# check once on the test set
proba = logreg.fit(X_train, y_train).predict_proba(X_test)[:, 1]
print(cost(y_test, proba, t_best), cost(y_test, proba, 0.5))   # 109200 127200
print(LOST * y_test.sum(), OFFER * len(y_test))               # 186800 176100: "no offers" and "offers to all"

0.25 329800 367500
109200 127200
186800 176100


The cost-based threshold saves €18,000 on 1,761 test customers compared with the default 0.5, and €77,600 compared with doing nothing. The empirical optimum (0.25) agrees with the formula *c / L*, a sign that the probabilities are well calibrated.

scikit-learn automates the search with **`TunedThresholdClassifierCV`**, which cross-validates the threshold for any scoring function:

In [5]:
from sklearn.metrics import make_scorer
from sklearn.model_selection import TunedThresholdClassifierCV

def gain(y_true, y_pred):                       # higher is better, so return minus the cost
    y_true = np.asarray(y_true)
    return -(OFFER * (y_pred == 1).sum() + LOST * ((y_true == 1) & (y_pred == 0)).sum())

tuned = TunedThresholdClassifierCV(logreg, scoring=make_scorer(gain), cv=5, random_state=0)
tuned.fit(X_train, y_train)
print(round(tuned.best_threshold_, 3))          # 0.295 (searched on a fine grid, cross-validated)
print(cost(y_test, proba, tuned.best_threshold_))   # 109500, close to the manual choice

0.295
109500


The worked example [05-tuned-decision-threshold.ipynb](05-tuned-decision-threshold.ipynb) (scikit-learn gallery) treats the same idea on the diabetes data.

**In practice.**
- Banks choose the cut-off of a credit score from the expected loss of a default and the expected profit of a good loan; the cut-off is revised when interest rates or loss rates change.
- Medical screening programmes set the positivity threshold of a test (for example the PSA level for prostate cancer or the faecal immunochemical test cut-off for bowel cancer) by weighing missed cancers against unnecessary follow-up examinations; different countries choose different cut-offs for the same test.

> **Caution.** Choosing the threshold on the test set is tuning on the test set (Session 7). Use out-of-fold predictions on the training data or a separate validation set, then evaluate once.

> **Tip.** Costs are often uncertain. Plot the total cost against the threshold: if the curve is flat around the optimum, the exact choice matters little; if it is steep, the cost assumptions deserve a second look with the business owner.

> 📖 **Read more:** [extras/08-cost-sensitive-learning.ipynb](extras/08-cost-sensitive-learning.ipynb) (scikit-learn gallery): cost-sensitive thresholds on German credit data and on 285,000 card transactions, where the cost of an error depends on the transaction amount.

## Class imbalance: class weights and resampling

**Concept.** A classification problem is **imbalanced** when one class is much rarer than the other. A learning algorithm that minimises the average loss over all rows gains little by getting the rare rows right, so it predicts the frequent class most of the time; lecture notebook 01 showed that "nobody churns" is already right for 73.5 % of the Telco customers. Three remedies change the *training*, not the data we evaluate on:

- A **class weight** multiplies the loss of every row of a class by a constant. `class_weight="balanced"` gives class *c* the weight w_c = n / (K · n_c), with *n* rows, *K* classes and *n_c* rows in class *c*. For the Telco data (n = 7,043, 1,869 churners): churners get 7,043 / (2 · 1,869) = 1.88, loyal customers 7,043 / (2 · 5,174) = 0.68.
- **Random undersampling** deletes rows of the frequent class until both classes are equally large (1,869 + 1,869 rows); **random oversampling** duplicates rows of the rare class (5,174 + 5,174 rows, 3,305 of them copies).
- **SMOTE** (Synthetic Minority Over-sampling Technique; Chawla et al., 2002) creates *new* minority rows on the line between a minority row **x** and one of its *k* nearest minority neighbours **x_nn**: **x_new = x + λ · (x_nn − x)**, with λ drawn uniformly between 0 and 1. Worked example: x = (2, 3), x_nn = (4, 7), λ = 0.25 gives x_new = (2.5, 4.0).

![SMOTE: original minority points, one point with its five nearest minority neighbours, and synthetic points placed on the lines between them](../figures/smote_illustration.png)

**Why it matters.** Imbalanced problems are the rule in practice: fraud, machine failures, diseases, churn and complaints are all rare compared with normal cases. All three remedies make the model predict the rare class more often: more recall, at the price of more false alarms. Class weights need no extra rows and no new library and are supported by most classifiers, so they are usually the first thing to try.

**How it works in Python.** For the comparisons of this and the next section we use all 7,043 customers with stratified 5-fold cross-validation. imbalanced-learn's samplers have a `fit_resample` method instead of `transform`, so they cannot be applied to test data by accident.

In [6]:
import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
telco = pd.read_csv(URL)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce").fillna(0)
y_churn = (telco["Churn"] == "Yes").astype(int)                   # 1 = customer left
X_churn = telco.drop(columns=["customerID", "Churn"])
cat_cols = X_churn.select_dtypes(exclude="number").columns.tolist()          # 15 categorical columns
print(y_churn.value_counts().to_dict(), round(y_churn.mean(), 3))  # {0: 5174, 1: 1869} 0.265

prep = make_column_transformer((OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
                               remainder=StandardScaler())
cv = StratifiedKFold(5, shuffle=True, random_state=0)


def report(name, pred):
    print(f"{name:20s} accuracy {accuracy_score(y_churn, pred):.3f}  recall of churners {recall_score(y_churn, pred):.3f}  "
          f"macro-F1 {f1_score(y_churn, pred, average='macro'):.3f}")


report("nobody churns", np.zeros(len(y_churn), dtype=int))
plain = make_pipeline(prep, LogisticRegression(max_iter=2000))
report("logistic regression", cross_val_predict(plain, X_churn, y_churn, cv=cv))     # threshold 0.5
# nobody churns        accuracy 0.735  recall of churners 0.000  macro-F1 0.424
# logistic regression  accuracy 0.803  recall of churners 0.546  macro-F1 0.733

from collections import Counter

from imblearn.over_sampling import SMOTENC, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.utils.class_weight import compute_class_weight

print(compute_class_weight("balanced", classes=np.array([0, 1]), y=y_churn).round(3))   # [0.681 1.884]
for sampler in [RandomUnderSampler(random_state=0), RandomOverSampler(random_state=0),
                SMOTENC(categorical_features=cat_cols, random_state=0)]:
    X_res, y_res = sampler.fit_resample(X_churn, y_churn)
    print(type(sampler).__name__, dict(Counter(y_res)))
# RandomUnderSampler {0: 1869, 1: 1869}
# RandomOverSampler {0: 5174, 1: 5174}
# SMOTENC {0: 5174, 1: 5174}   3,305 invented customers

weighted = make_pipeline(prep, LogisticRegression(max_iter=2000, class_weight="balanced"))
report("balanced weights", cross_val_predict(weighted, X_churn, y_churn, cv=cv))
# balanced weights     accuracy 0.748  recall of churners 0.803  macro-F1 0.719

{0: 5174, 1: 1869} 0.265


nobody churns        accuracy 0.735  recall of churners 0.000  macro-F1 0.424
logistic regression  accuracy 0.803  recall of churners 0.546  macro-F1 0.733


[0.681 1.884]
RandomUnderSampler {0: 1869, 1: 1869}
RandomOverSampler {0: 5174, 1: 5174}


SMOTENC {0: 5174, 1: 5174}
balanced weights     accuracy 0.748  recall of churners 0.803  macro-F1 0.719


At the default threshold the logistic regression finds 55 % of the churners. With balanced weights it finds 80 %, but its accuracy falls by 5 points: more churners are found at the price of many more false alarms. `SMOTENC` is the SMOTE variant for tables that mix numbers with categories: it interpolates the numeric columns and takes each category from the neighbours.

**In practice.**
- In the credit-card fraud dataset of the Université Libre de Bruxelles and Worldline (Dal Pozzolo et al., 2015), 492 of 284,807 transactions are fraudulent (0.17 %). A model that never flags fraud is 99.8 % accurate.
- XGBoost's documentation recommends the parameter `scale_pos_weight` (the ratio of negative to positive rows) for imbalanced binary problems such as fraud and churn; Session 10 uses it.

> **Warning.** Before correcting imbalance, ask whether the rare class matters for the decision. If you need well-calibrated probabilities, for example to forecast the number of churners, any correction distorts them (van den Goorbergh et al., 2022; last section).

> **Caution.** SMOTE invents data. It needs at least *k* + 1 rows per class, it works only on numbers (it cannot interpolate between two texts), and in high dimensions "between two neighbours" has little meaning. Always compare against class weights and against no correction.

> 📖 **Read more:** [extras/10-class-imbalance-in-depth.ipynb](extras/10-class-imbalance-in-depth.ipynb) (course notebook): the same methods in more depth, with partial sampling ratios, the SMOTE variants and the case of many rare classes.

## Resampling inside the cross-validation with imbalanced-learn

**Concept.** Resampling is part of *training*. It must be applied to the training folds of each cross-validation split, never to the validation fold and never to the test data, for two reasons:

1. **The validation data must look like reality.** Real customers churn at 26.5 %; a balanced validation set (50 %) makes every churn alarm look twice as likely to be right.
2. **Oversampling before splitting leaks.** If a copied (or SMOTE-interpolated) row lands in the training fold and its original in the validation fold, the model is tested on rows it has already seen.

scikit-learn's `Pipeline` cannot hold a sampler, because a sampler changes the number of rows. imbalanced-learn therefore provides its own `Pipeline` (`imblearn.pipeline.make_pipeline`) that calls `fit_resample` on samplers **only during `fit`** and skips them during `predict` and `score`. Inside `cross_validate` or `GridSearchCV`, every training fold is resampled and every validation fold stays untouched.

```mermaid
flowchart LR
    subgraph wrong["Wrong: resample first"]
        w1["All training data"] --> w2["Oversample"] --> w3["Split into folds"] --> w4["Copies in train<br/>and validation fold"]
    end
    subgraph right["Right: resample inside each fold"]
        r1["All training data"] --> r2["Split into folds"] --> r3["Oversample the<br/>training folds only"] --> r4["Validate on<br/>untouched fold"]
    end
```

**Why it matters.** The effect is not small. A random forest, which can memorise individual rows, looks excellent when the data were oversampled before the split. With one pipeline object, comparing methods also becomes a loop over candidates, and the sampling ratio or SMOTE's *k* can be tuned like any other hyperparameter.

**How it works in Python.** First the leak, then five strategies for the churn model, plus the alternative of the threshold section: no correction, but a lower decision threshold.

In [7]:
from imblearn.pipeline import make_pipeline as make_imb_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

rf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1)
X_dummies = pd.get_dummies(X_churn, dtype=int)                                     # the forest needs numbers

X_over, y_over = RandomOverSampler(random_state=0).fit_resample(X_dummies, y_churn)  # WRONG: before the split
wrong = cross_val_score(rf, X_over, y_over, cv=cv, scoring="f1").mean()

pipe = make_imb_pipeline(RandomOverSampler(random_state=0), rf)                     # RIGHT: inside each fold
right = cross_val_score(pipe, X_dummies, y_churn, cv=cv, scoring="f1").mean()
print(round(wrong, 3), round(right, 3))              # 0.899 0.577

0.899 0.577


0.899 is an illusion: the forest recognises copies of churners it has seen in training. 0.577 is the honest estimate, only a little above the forest without any correction (0.545 in our run).

In [8]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import make_pipeline as make_imb_pipeline
from sklearn.metrics import average_precision_score, precision_score, roc_auc_score


def lr(class_weight=None):
    return LogisticRegression(max_iter=2000, class_weight=class_weight)


candidates = {
    "no correction": make_imb_pipeline(prep, lr()),
    "undersampling": make_imb_pipeline(prep, RandomUnderSampler(random_state=0), lr()),
    "oversampling": make_imb_pipeline(prep, RandomOverSampler(random_state=0), lr()),
    "SMOTE": make_imb_pipeline(prep, SMOTE(random_state=0), lr()),
    "class weights": make_imb_pipeline(prep, lr("balanced")),
}


def scores(name, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {"method": name, "accuracy": accuracy_score(y_churn, pred), "precision": precision_score(y_churn, pred),
            "recall": recall_score(y_churn, pred), "f1": f1_score(y_churn, pred),
            "roc_auc": roc_auc_score(y_churn, proba), "pr_auc": average_precision_score(y_churn, proba)}


rows, probas = [], {}
for name, model in candidates.items():
    probas[name] = cross_val_predict(model, X_churn, y_churn, cv=cv, method="predict_proba")[:, 1]  # samplers: training folds only
    rows.append(scores(name, probas[name]))
rows.append(scores("threshold 0.3", probas["no correction"], threshold=0.3))        # no correction, lower threshold
print(pd.DataFrame(rows).set_index("method").round(3).to_string())
#                accuracy  precision  recall     f1  roc_auc  pr_auc
# method
# no correction     0.803      0.656   0.546  0.596    0.845   0.653
# undersampling     0.748      0.516   0.805  0.628    0.844   0.644
# oversampling      0.746      0.514   0.805  0.627    0.845   0.651
# SMOTE             0.754      0.523   0.800  0.633    0.845   0.652
# class weights     0.748      0.516   0.803  0.628    0.845   0.651
# threshold 0.3     0.766      0.542   0.769  0.636    0.845   0.653

               accuracy  precision  recall     f1  roc_auc  pr_auc
method                                                            
no correction     0.803      0.656   0.546  0.596    0.845   0.653
undersampling     0.748      0.516   0.805  0.628    0.844   0.644
oversampling      0.746      0.514   0.805  0.627    0.845   0.651
SMOTE             0.754      0.523   0.800  0.633    0.845   0.652
class weights     0.748      0.516   0.803  0.628    0.845   0.651
threshold 0.3     0.766      0.542   0.769  0.636    0.845   0.653


![Accuracy, precision, recall and F1 of the churn class for five imbalance strategies and for a lower decision threshold without correction](../figures/resampling_precision_recall.png)

An honest result. Every correction does what it promises: the recall of churners rises from 0.55 to about 0.80, at the price of precision (0.66 → 0.52) and accuracy (0.80 → 0.75). F1 of the churn class rises a little (0.60 → about 0.63). But ROC AUC and PR AUC do not change: the corrections do not make the model *rank* customers better, they only move the point at which it says "churn". Lowering the threshold of the uncorrected model to 0.3 reaches the same place (F1 0.636) without resampling, without invented customers and with probabilities that keep their meaning. SMOTE is not better than simply copying churners. For the churn task, the recommendation is: no resampling, choose the threshold from the costs of a missed churner and a wasted retention offer (threshold section above).

**In practice.**

- The imbalanced-learn library (Lemaître, Nogueira & Aridas, 2017) is a scikit-learn-contrib project under the MIT licence and the standard Python implementation of these methods; its examples compare samplers in pipelines as above.

> **Tip.** Report per-class precision and recall next to F1 or macro-F1. "With class weights we find 80 % of the churners instead of 55 %, but only half of the customers we call would have left" tells a retention team what the model does; "F1 rose by 0.03" does not.

> 📖 **Read more:** [extras/11-imbalanced-classes-impact.ipynb](extras/11-imbalanced-classes-impact.ipynb) and [extras/12-imblearn-pipeline.ipynb](extras/12-imblearn-pipeline.ipynb) (imbalanced-learn examples): the effect of imbalance and of each remedy on the adult census data, and a sampler inside an imbalanced-learn `Pipeline` with a classification report.

## Calibration of predicted probabilities

**Concept.** A classifier is **calibrated** if its probabilities mean what they say: among all customers who receive a churn probability of about 0.3, about 30 % actually churn. A **calibration curve** (reliability diagram) groups the test cases into bins by predicted probability and plots the mean predicted probability (x-axis) against the observed share of positives (y-axis). A calibrated model lies on the diagonal.

Two summary numbers:

- the **Brier score**, the mean squared difference between the predicted probability and the outcome (0 or 1); lower is better; always predicting the base rate 0.265 gives 0.265 · 0.735 ≈ 0.195;
- the **log-loss**, which punishes confident wrong predictions very strongly.

Logistic regression is usually well calibrated, because it is fitted by maximising the likelihood of the observed outcomes. Other models are not: **naive Bayes** (a classifier that multiplies per-feature probabilities as if features were independent) pushes probabilities towards 0 and 1, and boosted trees and some neural networks can be over- or under-confident. Models trained with re-weighted or resampled classes (previous sections) are systematically miscalibrated. **Recalibration** fits a mapping from the model's scores to calibrated probabilities on held-out data: **Platt scaling** (a logistic curve, `method="sigmoid"`) or **isotonic regression** (a monotone step function, `method="isotonic"`), both in `CalibratedClassifierCV`.

![Calibration curves on the Telco test customers. Logistic regression lies close to the diagonal; naive Bayes predicts probabilities near 0 or 1 for most customers and is far from the diagonal; after isotonic recalibration it is close to the diagonal again.](../figures/calibration_curve.png)

**Why it matters.** The cost rule *p > c / L*, expected-revenue calculations ("expected churners next month = sum of probabilities") and risk communication to customers or patients all assume calibrated probabilities. A model can rank well (high AUC) and still be badly calibrated; AUC does not detect it.

**How it works in Python.**

In [9]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.naive_bayes import GaussianNB

nb = make_pipeline(pre, GaussianNB()).fit(X_train, y_train)
nb_cal = CalibratedClassifierCV(make_pipeline(pre, GaussianNB()), method="isotonic", cv=5).fit(X_train, y_train)

for name, m in [("logistic regression", logreg), ("naive Bayes", nb), ("naive Bayes, recalibrated", nb_cal)]:
    q = m.predict_proba(X_test)[:, 1]
    frac, mean_p = calibration_curve(y_test, q, n_bins=5, strategy="quantile")
    print(f"{name:26s} Brier {brier_score_loss(y_test, q):.3f}  mean p {q.mean():.3f}")
    print("   predicted", mean_p.round(2), " observed", frac.round(2))
# logistic regression        Brier 0.138  mean p 0.265
#    predicted [0.01 0.06 0.19 0.4  0.66]  observed [0.01 0.07 0.2  0.41 0.64]
# naive Bayes                Brier 0.278  mean p 0.471
#    predicted [0.   0.   0.36 1.   1.  ]  observed [0.07 0.05 0.19 0.42 0.6 ]
# naive Bayes, recalibrated  Brier 0.150  mean p 0.266
#    predicted [0.06 0.08 0.17 0.38 0.68]  observed [0.05 0.08 0.2  0.44 0.59]

logistic regression        Brier 0.138  mean p 0.265
   predicted [0.01 0.06 0.19 0.4  0.66]  observed [0.01 0.07 0.2  0.41 0.64]
naive Bayes                Brier 0.278  mean p 0.471
   predicted [0.   0.   0.36 1.   1.  ]  observed [0.07 0.05 0.19 0.42 0.6 ]
naive Bayes, recalibrated  Brier 0.150  mean p 0.266
   predicted [0.06 0.08 0.17 0.38 0.68]  observed [0.05 0.08 0.2  0.44 0.59]


Naive Bayes predicts "certain churn" (p ≈ 1) for many customers of whom only about half churn; its mean probability (0.47) is far above the true churn rate (0.265). Recalibration repairs this. The scikit-learn gallery notebook [extras/09-calibration-curve.ipynb](extras/09-calibration-curve.ipynb) compares more models.

**In practice.**
- Weather forecasting: the US National Weather Service's probability-of-precipitation forecasts have been studied for calibration since the 1970s (Murphy & Winkler, 1977); "30 % chance of rain" is meant to come true about 30 % of the time.
- Clinical risk scores (for example cardiovascular risk calculators) are checked for calibration in each new population; a score developed in one country often over- or under-predicts risk in another and is recalibrated before use.

> **Warning.** Recalibrate on data the model was not trained on. `CalibratedClassifierCV(cv=5)` does this internally with cross-validation; never fit the calibration mapping on the test set.

**The case study: a model trained with class weights.** The previous sections showed that weighting the rare class (`class_weight="balanced"`) is one way to find more churners. It changes the probabilities, not the ranking:

In [10]:
weighted = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced")).fit(X_train, y_train)
weighted_cal = CalibratedClassifierCV(make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced")),
                                      method="sigmoid", cv=5).fit(X_train, y_train)
t_rule = OFFER / LOST                                  # 0.25, valid only for calibrated probabilities
for name, m in [("logistic regression", logreg), ("class weights", weighted), ("class weights, recalibrated", weighted_cal)]:
    q = m.predict_proba(X_test)[:, 1]
    print(f"{name:28s} AUC {roc_auc_score(y_test, q):.3f}  Brier {brier_score_loss(y_test, q):.3f}  "
          f"expected churners {q.sum():.0f}  cost at c/L {cost(y_test, q, t_rule)}")
# logistic regression          AUC 0.844  Brier 0.138  expected churners 466  cost at c/L 109200
# class weights                AUC 0.844  Brier 0.165  expected churners 725  cost at c/L 116400
# class weights, recalibrated  AUC 0.844  Brier 0.138  expected churners 465  cost at c/L 108700

logistic regression          AUC 0.844  Brier 0.138  expected churners 466  cost at c/L 109200
class weights                AUC 0.844  Brier 0.165  expected churners 725  cost at c/L 116400
class weights, recalibrated  AUC 0.844  Brier 0.138  expected churners 465  cost at c/L 108700


The test set has 467 churners. The weighted model ranks the customers exactly as well (same AUC) but predicts 725 churners, and the rule *p > c / L* applied to its probabilities costs €7,200 more than the same rule on calibrated probabilities. After recalibration the rule works again. A threshold tuned on out-of-fold predictions (`TunedThresholdClassifierCV` above) also compensates, because it looks at the costs, not at the probabilities; it finds 0.52 for the weighted model.

> **Note.** For three or more classes calibration is checked per class (one-vs-rest) or for the **confidence**, the highest predicted probability of a case: among cases with a confidence of about 0.4, is the prediction right about 40 % of the time? Session 14 uses the confidence of a text classifier to decide which cases it settles on its own and which it passes to a person.

## Check your understanding

1. Explain the meaning of AUC = 0.84 in one sentence without using the word "curve".
2. For a fraud model with 0.1 % positives, why can ROC AUC be 0.98 while precision at 50 % recall is only 0.10?
3. A retention offer costs €50 and a lost customer costs €500. Which threshold follows from *p > c / L*? What assumption does this rule make?
4. Why did oversampling before cross-validation give an F1 of 0.90 instead of 0.58 with a random forest?
5. A model trained with `class_weight="balanced"` predicts 725 churners among 1,761 customers, of whom 467 churn. Is its ROC AUC necessarily worse than that of the unweighted model? What happens if you apply the rule *p > c / L* to its probabilities?

## Further reading

- scikit-learn developers (2025). *Tuning the decision threshold for class prediction* and *Probability calibration*. https://scikit-learn.org/stable/modules/classification_threshold.html · https://scikit-learn.org/stable/modules/calibration.html
- Saito, T. & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE*, 10(3), e0118432. https://doi.org/10.1371/journal.pone.0118432
- Elkan, C. (2001). The foundations of cost-sensitive learning. *Proceedings of IJCAI 2001*, 973–978. https://cseweb.ucsd.edu/~elkan/rescale.pdf
- imbalanced-learn developers (2025). *Common pitfalls and recommended practices*. https://imbalanced-learn.org/stable/common_pitfalls.html
- van den Goorbergh, R., van Smeden, M., Timmerman, D. & Van Calster, B. (2022). The harm of class imbalance corrections for risk prediction models. *JAMIA*, 29(9), 1525–1534. https://doi.org/10.1093/jamia/ocac093
- Van Calster, B., McLernon, D. J., van Smeden, M., Wynants, L. & Steyerberg, E. W. (2019). Calibration: the Achilles heel of predictive analytics. *BMC Medicine*, 17, 230. https://doi.org/10.1186/s12916-019-1466-7